# Tema alesa

### 18. Sisteme automate bazate pe procesarea semnalelor EEG și învățare automată

Actualizat pentru 2026: extins cu BCI bazată pe imaginație motorie pentru controlul unei proteze.
Obiectiv: Obiectivul acestui proiect este dezvoltarea unei interfețe creier-calculator (BCI) care
achiziționează semnale EEG, clasifică intenția utilizatorului și transmite comanda identificată
către un dispozitiv controlat. Aplicația principală propusă este controlul unei proteze de
membru superior prin imaginația motorie a mâinii stângi și a mâinii drepte, cu un model de
clasificare care generalizează la subiecți noi fără calibrare individuală. Sunt acceptate și alte
aplicații (control robotic, conducere autonomă, automatizarea locuinței).

__Setul de date 1 – setul de date de imaginație motorie WBCIC-SHU (Yang et al., 2025)__

Înregistrări EEG multi-subiect și multi-sesiune ale imaginației motorii a mâinii stângi și a
mâinii drepte, cu 53 de canale; utilizat în ediția anterioară pentru definirea protocolului
de antrenare/validare (scenarii intra-sesiune, inter-sesiune și inter-subiect).

__Placa OpenBCI Cyton (Daisy) cu 16 canale, împreună cu casca Ultracortex Mark IV__

achiziție cu BrainFlow (interfețe Python/MATLAB) sau cu OpenBCI GUI.
• Documentație: docs.openbci.com/Cyton/CytonLanding și brainflow.readthedocs.io;
dosarele proiectelor anterioare conțin exemple de înregistrări (Fatigue state, articol
despre imaginație motorie, EEGLAB).

__Metodologie:__ 

Preprocesare în MATLAB/EEGLAB sau Python (filtrare trece-bandă și notch,
re-referențiere, segmentare în epoci, eliminarea liniei de bază, re-eșantionare); analiză
temporală, spațială și spectrală pentru confirmarea calității datelor; compararea CSP + SVM,
FBCSP + SVM, EEGNet și DeepConvNet în scenarii cu dificultate crescândă (intra-sesiune,
inter-sesiune, inter-subiect); selectarea arhitecturii cu cel mai bun compromis între
performanță și cost computațional; validare pe subiecți complet izolați de antrenare; reducerea
numărului de canale pentru a corespunde căștii disponibile; interfață cu utilizatorul
(monitorizarea predicției și a acurateței) și transmiterea comenzii către dispozitiv (de exemplu,
o proteză acționată de servomotoare prin Arduino).

__Rezultate așteptate:__

• Un clasificator de imaginație motorie independent de subiect, validat pe subiecți izolați.
• Un studiu al influenței numărului de canale asupra acurateței.
• Un lanț demonstrativ complet: EEG → preprocesare → clasificator → interfață cu
utilizatorul → proteză/robot.

In [4]:
%pip install -r ../requirements.txt

  Using cached moabb-1.7.2-py3-none-any.whl.metadata (15 kB)
  Using cached mne-1.13.2-py3-none-any.whl.metadata (17 kB)
  Using cached pymatreader-1.3.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached pyriemann-0.12-py2.py3-none-any.whl.metadata (10.0 kB)
  Using cached nemar_py-0.3.3-py3-none-any.whl.metadata (8.4 kB)
  Using cached pooch-1.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached remotezip-0.12.6-py3-none-any.whl.metadata (7.3 kB)
  Using cached coverage-7.17.2-cp313-cp313-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl.metadata (8.1 kB)
  Using cached memory_profiler-0.61.0-py3-none-any.whl.metadata (20 kB)
  Using cached EDFlib_Python-1.0.8-py3-none-any.whl.metadata (1.3 kB)
  Using cached edfio-0.4.18-py3-none-any.whl.metadata (4.0 kB)
  Using cached mne_bids-0.20.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached pybv-0.8.1-py3-none-any.whl.metadata (14 kB)
  Using cached pycountry-26.2.16-py3-none-any.whl.metadata (12 kB)
  Using cached docstring

In [5]:
# Import necessary libraries

import numpy as np
import pandas as pd
import warnings as wn
import matplotlib.pyplot as plt

from sklearn import *
from pathlib import Path
from moabb.datasets import Yang2025
from moabb.paradigms import LeftRightImagery

/home/john/.anaconda3/lib/python3.13/importlib/__init__.py:88: FutureWarning: `optuna.integration.sklearn` has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0. Use `optuna_integration.sklearn` instead.
  return _bootstrap._gcd_import(name[level:], package, level)


In [6]:
# Ignore all warnings
wn.filterwarnings('ignore')

In [7]:
# Create general variables

_data_path = '../data'
_limit_subjects = 5
_data_dir = Path(_data_path)
_train_path = f'{_data_path}/participants.tsv'

In [8]:
# Read the dataset
dataset = Yang2025()
print(f'Available: {dataset.subject_list}')

Available: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51]


In [9]:
# Load some pacients
subjects = dataset.subject_list[:_limit_subjects] 

In [10]:
# Set the paradigm dataset
paradigm = LeftRightImagery(fmin=8, fmax=32)

In [ ]:
# Create the dataframes
X, y, metadata = paradigm.get_data(dataset=dataset, subjects=subjects)

Attempting to create new mne-python configuration file:
/home/john/.mne/mne-python.json
Could not read the /home/john/.mne/mne-python.json json file during the writing. Assuming it is empty. Got: Expecting value: line 1 column 1 (char 0)
This is nemar-py 0.3.3.
Preparing to download nm000348 from https://data.nemar.org/
Retrieving 6 of 2231 manifest files (16 concurrent downloads).


Overall:   0%|          | 0.00/248k [00:00<?, ?B/s]

Finished downloading nm000348 v1.0.5.
This is nemar-py 0.3.3.
Preparing to download nm000348 from https://data.nemar.org/
Retrieving 26 of 2231 manifest files (16 concurrent downloads).


S3:   0%|          | 0.00/3.16G [00:00<?, ?B/s]

Overall:   0%|          | 0.00/9.91k [00:00<?, ?B/s]

Finished downloading nm000348 v1.0.5.
This is nemar-py 0.3.3.
Preparing to download nm000348 from https://data.nemar.org/
Retrieving 26 of 2231 manifest files (16 concurrent downloads).


... (9 outputs left out)

S3:   0%|          | 0.00/3.15G [00:00<?, ?B/s]

Overall:   0%|          | 0.00/9.95k [00:00<?, ?B/s]

Finished downloading nm000348 v1.0.5.


  0%|                                              | 0.00/65.6G [00:00<?, ?B/s]

In [ ]:
# Show the shape of the data
x_rows, x_cols = X.shape
y_rows, y_ros = y.shape
print('Input Rows: ', x_rows)
print('Input Columns: ', x_cols)
print('Output Rows: ', y_rows)
print('Output Columns: ', y_cols)

Rows:  30591
Columns:  15


In [20]:
# Display the first n rows
master.head(n=10)

,onset,duration,trial_type,value,sample,participant_id,session_id,age,sex,hand,weight,height,group,species,bci_experience
0,233.465,4.0,left_hand,1,233465,sub-1,ses-0,29,F,R,NaN,NaN,healthy,human,naive
1,241.465,4.0,left_hand,1,241465,sub-1,ses-0,29,F,R,NaN,NaN,healthy,human,naive
2,249.465,4.0,right_hand,2,249465,sub-1,ses-0,29,F,R,NaN,NaN,healthy,human,naive
3,257.465,4.0,right_hand,2,257465,sub-1,ses-0,29,F,R,NaN,NaN,healthy,human,naive
4,265.464,4.0,right_hand,2,265464,sub-1,ses-0,29,F,R,NaN,NaN,healthy,human,naive
5,273.463,4.0,right_hand,2,273463,sub-1,ses-0,29,F,R,NaN,NaN,healthy,human,naive
6,281.462,4.0,left_hand,1,281462,sub-1,ses-0,29,F,R,NaN,NaN,healthy,human,naive
7,289.462,4.0,left_hand,1,289462,sub-1,ses-0,29,F,R,NaN,NaN,healthy,human,naive
8,297.462,4.0,right_hand,2,297462,sub-1,ses-0,29,F,R,NaN,NaN,healthy,human,naive
9,305.462,4.0,left_hand,1,305462,sub-1,ses-0,29,F,R,NaN,NaN,healthy,human,naive


In [21]:
# Describe the dataframe
master.describe()

,onset,duration,value,sample,age,weight,height
count,30591.000000,30591.0,30591.000000,3.059100e+04,30591.0,0.0,0.0
mean,1251.176499,4.0,1.500016,1.251176e+06,29.0,NaN,NaN
std,631.994997,0.0,0.500008,6.319950e+05,0.0,NaN,NaN
min,142.343000,4.0,1.000000,1.423430e+05,29.0,NaN,NaN
25%,724.951000,4.0,1.000000,7.249510e+05,29.0,NaN,NaN
50%,1238.960000,4.0,2.000000,1.238960e+06,29.0,NaN,NaN
75%,1755.461000,4.0,2.000000,1.755461e+06,29.0,NaN,NaN
max,3239.109000,4.0,2.000000,3.239109e+06,29.0,NaN,NaN


In [26]:
# Show more infos about the dataset
master.info()

<class 'pandas.DataFrame'>
RangeIndex: 30591 entries, 0 to 30590
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   onset           30591 non-null  float64
 1   duration        30591 non-null  float64
 2   trial_type      30591 non-null  str    
 3   value           30591 non-null  int64  
 4   sample          30591 non-null  int64  
 5   participant_id  30591 non-null  str    
 6   session_id      30591 non-null  str    
 7   age             30591 non-null  int64  
 8   sex             30591 non-null  str    
 9   hand            30591 non-null  str    
 10  weight          0 non-null      float64
 11  height          0 non-null      float64
 12  group           30591 non-null  str    
 13  species         30591 non-null  str    
 14  bci_experience  30591 non-null  str    
dtypes: float64(4), int64(3), str(8)
memory usage: 3.5 MB


In [30]:
# Unique class for each column
object_columns = master.select_dtypes(include=['str']).columns
for col in object_columns:
    print(f'{col}: {master[col].nunique()} values \t\t {master[col].unique()}')

trial_type: 2 values 		 <StringArray>
['left_hand', 'right_hand']
Length: 2, dtype: str
participant_id: 51 values 		 <StringArray>
[ 'sub-1', 'sub-10', 'sub-11', 'sub-12', 'sub-13', 'sub-14', 'sub-15',
 'sub-16', 'sub-17', 'sub-18', 'sub-19',  'sub-2', 'sub-20', 'sub-21',
 'sub-22', 'sub-23', 'sub-24', 'sub-25', 'sub-26', 'sub-27', 'sub-28',
 'sub-29',  'sub-3', 'sub-30', 'sub-31', 'sub-32', 'sub-33', 'sub-34',
 'sub-35', 'sub-36', 'sub-37', 'sub-38', 'sub-39',  'sub-4', 'sub-40',
 'sub-41', 'sub-42', 'sub-43', 'sub-44', 'sub-45', 'sub-46', 'sub-47',
 'sub-48', 'sub-49',  'sub-5', 'sub-50', 'sub-51',  'sub-6',  'sub-7',
  'sub-8',  'sub-9']
Length: 51, dtype: str
session_id: 3 values 		 <StringArray>
['ses-0', 'ses-1', 'ses-2']
Length: 3, dtype: str
sex: 2 values 		 <StringArray>
['F', 'M']
Length: 2, dtype: str
hand: 1 values 		 <StringArray>
['R']
Length: 1, dtype: str
group: 1 values 		 <StringArray>
['healthy']
Length: 1, dtype: str
species: 1 values 		 <StringArray>
['human']
Leng